# Fundamentos de Ingeniería de Datos
## Transformación: de EXTRACTED a TRANSFORMED

```
INGESTA            EXTRACCIÓN              TRANSFORMACIÓN (hoy)
Kaggle → raw_df1 → extracted_df1    →    diagnosticar · limpiar · tipar · enriquecer
        (SQLite)    (SQLite)              → df_final (en memoria) + bitácora
```

**Tema 2.2 — Transformaciones:** limpieza, normalización, tipado, *joins* y enriquecimiento.

### ¿Qué es la capa TRANSFORMED?

Son los datos de `extracted_df1` **limpios, con tipos correctos y listos para analizar**. Es la primera capa en la que **sí cambiamos** los datos.

| Capa | Qué contiene | En la industria (*medallion*) |
|---|---|---|
| RAW | Datos tal como llegaron | Bronze |
| EXTRACTED | Los mismos datos + metadatos de extracción | Bronze |
| **TRANSFORMED** | **Datos limpios, tipados y enriquecidos** | **Silver** |
| SERVING | Modelo para análisis (Unidad 3) | Gold |

> **Regla de oro:** cada capa lee **solo** de la anterior. Este notebook lee `extracted_df1` (nunca `raw_df1`) en **modo solo lectura**.

### Configuración del entorno

In [ ]:
# ── Montar Google Drive ────────────────────────────────────────────────────

#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
sys.path.append('/Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Utilidades')

from utils import PROYECTO_BASE, DB_PATH, inspeccionar_dataset, listar_tablas

print()
print('utils.py importado correctamente')
print(f'Base de datos: {DB_PATH}')
print()


utils.py importado correctamente
Base de datos: /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Base_de_datos/base_datos.db



In [7]:
# ── Librerías, rutas y nombres ─────────────────────────────────────────────

import sqlite3
import pandas as pd
from datetime import datetime




TABLA_EXTRACTED   = 'extracted_df1'        # origen  (SQLite)
NOMBRE_DESTINO    = 'transformed_df1'      # nombre de la capa limpia (se cargará en la siguiente parte)
VERSION_TRANSFORM = 'v1'                   # súbela si cambias las reglas de limpieza


print()
print(f'Origen : {TABLA_EXTRACTED}  (SQLite, solo lectura)')
print()


Origen : extracted_df1  (SQLite, solo lectura)



---
## PASO 1 — Leer el origen: `extracted_df1`

Leemos en **solo lectura** y separamos las columnas en dos grupos:

- **Columnas de negocio:** las del dataset (las vamos a limpiar).
- **Metadatos** (empiezan con `_`): **no se tocan**. `_row_hash` nos permitirá rastrear cada fila limpia hasta su origen (*lineage*).

In [8]:
# ── ¿Qué tablas hay en el .db? ─────────────────────────────────────────────

listar_tablas(DB_PATH);


  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  extracted_df1                       extracted_        16,598     15
  raw_df1                             raw_              16,598     11
  raw_sp2                             raw_                 500     12
  raw_sp3                             raw_                 646     17


In [5]:
# ── Leer extracted_df1 en modo solo lectura ────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()



COLS_META    = [c for c in df_ext.columns if c.startswith('_')]
COLS_NEGOCIO = [c for c in df_ext.columns if not c.startswith('_')]



print()
print(f'Columnas de negocio : {COLS_NEGOCIO}')
print(f'Metadatos           : {COLS_META}')

inspeccionar_dataset(df_ext, nombre=TABLA_EXTRACTED)


Columnas de negocio : ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
Metadatos           : ['_batch_id', '_extracted_at', '_source_table', '_row_hash']

  DATASET: extracted_df1
  Filas    : 16,598
  Columnas : 15

  Tipos de datos y valores nulos:
    Rank                         int64        nulos: 0 (0.0%)
    Name                         str          nulos: 0 (0.0%)
    Platform                     str          nulos: 0 (0.0%)
    Year                         float64      nulos: 271 (1.6%)
    Genre                        str          nulos: 0 (0.0%)
    Publisher                    str          nulos: 58 (0.3%)
    NA_Sales                     float64      nulos: 0 (0.0%)
    EU_Sales                     float64      nulos: 0 (0.0%)
    JP_Sales                     float64      nulos: 0 (0.0%)
    Other_Sales                  float64      nulos: 0 (0.0%)
    Global_Sales                 float64      nu

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,_batch_id,_extracted_at,_source_table,_row_hash
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,2,2026-10-21T18:04:50,raw_df1,4afd4e8c1f96d38e
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,2,2026-10-21T18:04:50,raw_df1,3ee71d62807f116a
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,2,2026-10-21T18:04:50,raw_df1,688fb7c1f6ec1137


---
## PASO 2 — Diagnóstico de calidad (antes)

**No se limpia lo que no se ha medido.** Antes de transformar, medimos los problemas. Al final repetimos la medición y comparamos: esa comparación va en tu **reporte de validación**.

| Dimensión de calidad | Pregunta | Cómo la medimos |
|---|---|---|
| **Completitud** | ¿Faltan datos? | Nulos por columna |
| **Unicidad** | ¿Hay registros repetidos? | Duplicados exactos y por llave |
| **Validez** | ¿Los valores tienen sentido? | Rangos (años, ventas ≥ 0) |
| **Consistencia** | ¿Los datos cuadran entre sí? | ¿`Global_Sales` = suma de regiones? |

**Llave de negocio:** el conjunto de columnas que identifica a un registro. En vgsales, un juego se identifica por **nombre + plataforma + año** (*Wii Sports* en Wii no es el mismo registro que *Wii Sports* en otra consola).

In [9]:
# ── Reglas del dataset (ajústalas a TU dataset) ────────────────────────────

COLS_TEXTO    = ['Name', 'Platform', 'Genre', 'Publisher']       # columnas de texto
COLS_REGIONES = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
COL_GLOBAL    = 'Global_Sales'
COLS_VENTAS   = COLS_REGIONES + [COL_GLOBAL]                     # las 5 columnas de ventas

ANIO_MIN   = 1980        # primera consola comercial del dataset
ANIO_MAX   = 2016        # el dataset se recolectó en 2016: años posteriores son errores

TOLERANCIA = 0.05        # millones; diferencia aceptable por redondeo

In [10]:
# ── Nulos por columna ──────────────────────────────────────────────────────

nulos = df_ext[COLS_NEGOCIO].isna().sum()

print()
print(f'Total de celdas nulas: {nulos.sum():,}')
print()

nulos


Total de celdas nulas: 329



Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

In [11]:
# ── Registros repetidos ────────────────────────────────────────────────────

# Exactos: todas las columnas del dataset son iguales
dup_exactos = df_ext.duplicated(subset=COLS_NEGOCIO).sum()

print()
print(f'Duplicados exactos: {dup_exactos}')
print()

# Ver los registros repetidos
repetidos = df_ext[df_ext.duplicated(subset=COLS_NEGOCIO, keep=False)]
repetidos[COLS_NEGOCIO]


Duplicados exactos: 0



,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales


In [12]:
# Años fuera de rango (los nulos no cuentan: ya se revisaron aparte)

anio_invalido = ~df_ext['Year'].between(ANIO_MIN, ANIO_MAX) & df_ext['Year'].notna()

# Filas con al menos una venta negativa
venta_negativa = (df_ext[COLS_VENTAS] < 0).any(axis=1)

print()
print(f'Años fuera de [{ANIO_MIN}, {ANIO_MAX}] : {anio_invalido.sum()}')
print(f'Filas con ventas negativas    : {venta_negativa.sum()}')
print()

# Ver los registros con año inválido
df_ext.loc[anio_invalido, COLS_NEGOCIO]


Años fuera de [1980, 2016] : 4
Filas con ventas negativas    : 0



,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
5957,5959,Imagine: Makeup Artist,DS,2020.0,Simulation,Ubisoft,0.27,0.0,0.00,0.02,0.29
14390,14393,Phantasy Star Online 2 Episode 4: Deluxe Package,PS4,2017.0,Role-Playing,Sega,0.00,0.0,0.03,0.00,0.03
16241,16244,Phantasy Star Online 2 Episode 4: Deluxe Package,PSV,2017.0,Role-Playing,Sega,0.00,0.0,0.01,0.00,0.01
16438,16441,Brothers Conflict: Precious Baby,PSV,2017.0,Action,Idea Factory,0.00,0.0,0.01,0.00,0.01


In [11]:
# ── Consistencia: ¿los datos cuadran entre sí? ─────────────────────────────

# Suma de las 4 regiones y diferencia contra las ventas globales
suma_regiones = df_ext[COLS_REGIONES].sum(axis=1)
diferencia    = (df_ext['Global_Sales'] - suma_regiones).abs()

inconsistente = diferencia > TOLERANCIA

print()
print(f'Filas inconsistentes (diferencia > {TOLERANCIA}) : {inconsistente.sum()}')
print(f'Diferencia máxima encontrada            : {diferencia.max():.2f}')
print()

# Ver los registros inconsistentes
df_ext.loc[inconsistente, ['Name', 'Platform'] + COLS_REGIONES + ['Global_Sales']]


Filas inconsistentes (diferencia > 0.05) : 0
Diferencia máxima encontrada            : 0.02



,Name,Platform,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales


In [12]:
# ── Resumen del diagnóstico (línea base para comparar después) ─────────────

calidad_antes = pd.Series({
    'filas'                  : len(df_ext),
    'celdas_nulas'           : nulos.sum(),
    'duplicados_exactos'     : dup_exactos,
    'anios_fuera_de_rango'   : anio_invalido.sum(),
    'ventas_negativas'       : venta_negativa.sum(),
    'ventas_inconsistentes'  : inconsistente.sum(),
}, name='antes')

calidad_antes.to_frame()

,antes
filas,16598
celdas_nulas,329
duplicados_exactos,0
anios_fuera_de_rango,4
ventas_negativas,0
ventas_inconsistentes,0


---
## PASO 3 — Transformaciones

Ahora sí **cambiamos los datos**, atacando lo que encontró el diagnóstico del Paso 2. Aplicamos las reglas **una por una** sobre una copia (`df`) y anotamos cada una en una **bitácora**: qué se hizo y cuántas filas cambió. La bitácora es el *lineage* de las transformaciones y va tal cual en tu reporte.

| # | Transformación | Tipo | ¿Qué problema del Paso 2 resuelve? |
|---|---|---|---|
| 3.1 | Nombres de columnas en `snake_case` | Normalización | Convención de nombres |
| 3.2 | Limpiar texto | Normalización | Espacios que separan valores iguales |
| 3.3 | Tratar nulos | Limpieza | **Completitud** |
| 3.4 | Quitar duplicados exactos | Limpieza | **Unicidad** |
| 3.5 | Rangos y consistencia | Limpieza | **Validez** y **consistencia** |
| 3.6 | Convertir tipos de dato | Tipado | Tipos incorrectos (año como decimal) |
| 3.7 | Enriquecer con un catálogo (*join*) | Enriquecimiento | — |
| 3.8 | Columnas derivadas | Enriquecimiento | — |
| 3.9 | Metadatos de transformación | Lineage | — |

In [13]:
# ── Bitácora de transformaciones ───────────────────────────────────────────

bitacora = []

def registrar(paso, descripcion, filas_antes, filas_despues, afectadas):

    '''Anota un paso en la bitácora y lo muestra'''

    bitacora.append(

        {
        'paso'            : paso,
        'descripcion'     : descripcion,
        'filas_antes'     : filas_antes,
        'filas_despues'   : filas_despues,
        'filas_afectadas' : int(afectadas),
    })


    print(f'[{paso}] {descripcion}')
    print(f'       filas: {filas_antes:,} → {filas_despues:,}   (afectadas: {int(afectadas):,})')



df = df_ext.copy()        # trabajamos sobre una copia: df_ext queda intacto para comparar


print()
print(f'Punto de partida: {len(df):,} filas')
print()


Punto de partida: 16,598 filas



### 3.1 Nombres de columnas en `snake_case`

Una convención única (minúsculas, sin espacios, palabras separadas por `_`) evita errores al escribir SQL: `Global_Sales`, `global_sales` y `"Global Sales"` dejan de ser un problema.

Como las columnas cambian de nombre, las listas de reglas del Paso 2 también se actualizan con la misma función.

In [14]:
# ── 3.1 Renombrar columnas ─────────────────────────────────────────────────

def a_snake_case(columnas):

    '''Minúsculas, sin espacios a los lados y con _ en lugar de espacios.'''

    return list(pd.Index(columnas).str.strip().str.lower().str.replace(' ', '_'))


df.columns = a_snake_case(df.columns)

# Las listas de reglas usan ahora los nombres nuevos

COLS_NEGOCIO  = a_snake_case(COLS_NEGOCIO)
COLS_TEXTO    = a_snake_case(COLS_TEXTO)
COLS_REGIONES = a_snake_case(COLS_REGIONES)
COL_GLOBAL    = COL_GLOBAL.lower()
COLS_VENTAS   = COLS_REGIONES + [COL_GLOBAL]

registrar('3.1', 'Nombres de columnas a snake_case', len(df), len(df), 0)

pd.DataFrame({'antes': df_ext.columns, 'después': df.columns})

[3.1] Nombres de columnas a snake_case
       filas: 16,598 → 16,598   (afectadas: 0)


,antes,después
0,Rank,rank
1,Name,name
2,Platform,platform
3,Year,year
4,Genre,genre
5,Publisher,publisher
6,NA_Sales,na_sales
7,EU_Sales,eu_sales
8,JP_Sales,jp_sales
9,Other_Sales,other_sales


### 3.2 Limpiar texto

Espacios al inicio o al final (`'Nintendo '`) o dobles espacios hacen que dos valores iguales se vean distintos al agrupar o al hacer *joins*. `apply()` ejecuta la función de limpieza en cada columna de `COLS_TEXTO`.

In [15]:
# ── 3.2 Limpiar columnas de texto ──────────────────────────────────────────

def limpiar_texto(columna):

    '''Quita espacios al inicio y al
     final, y deja un solo espacio entre palabras.'''


    return columna.str.strip().str.replace(r'\s+', ' ', regex=True)




texto_antes = df[COLS_TEXTO].copy()

df[COLS_TEXTO] = df[COLS_TEXTO].apply(limpiar_texto)

# Celdas que cambiaron (fillna('') evita contar los nulos como cambios)

cambiadas = (df[COLS_TEXTO].fillna('') != texto_antes.fillna('')).sum().sum()


print()
registrar('3.2', f'Quitar espacios extra en {COLS_TEXTO}', len(df), len(df), cambiadas)
print()



[3.2] Quitar espacios extra en ['name', 'platform', 'genre', 'publisher']
       filas: 16,598 → 16,598   (afectadas: 5)



### 3.3 Tratar nulos (completitud)

No hay una regla única: **cada columna se decide según su significado** y la decisión se documenta.

| Columna | Opciones | Decisión | Por qué |
|---|---|---|---|
| `publisher` | eliminar / imputar | **Imputar `'Unknown'`** | El juego existe y sus ventas son válidas; solo falta el dato de la editorial |
| `year` | eliminar / imputar con la mediana / dejar nulo | **Eliminar la fila** | Inventar un año de lanzamiento sería un dato falso, y sin año el juego no sirve para analizar tendencias en el tiempo. Son pocas filas (revisa el Paso 2) |

> En tu dataset las decisiones pueden ser otras. Lo importante es **justificarlas** en el reporte.

In [16]:
# ── 3.3 Nulos ──────────────────────────────────────────────────────────────

# publisher → 'Unknown'
nulos_pub = df['publisher'].isna().sum()
df['publisher'] = df['publisher'].fillna('Unknown')
registrar('3.3a', "publisher nulo → 'Unknown'", len(df), len(df), nulos_pub)

# year nulo → eliminar fila
antes = len(df)
df = df.dropna(subset=['year'])
registrar('3.3b', 'Eliminar filas sin year', antes, len(df), antes - len(df))

[3.3a] publisher nulo → 'Unknown'
       filas: 16,598 → 16,598   (afectadas: 58)
[3.3b] Eliminar filas sin year
       filas: 16,598 → 16,327   (afectadas: 271)


In [17]:
print(bitacora)

[{'paso': '3.1', 'descripcion': 'Nombres de columnas a snake_case', 'filas_antes': 16598, 'filas_despues': 16598, 'filas_afectadas': 0}, {'paso': '3.2', 'descripcion': "Quitar espacios extra en ['name', 'platform', 'genre', 'publisher']", 'filas_antes': 16598, 'filas_despues': 16598, 'filas_afectadas': 5}, {'paso': '3.3a', 'descripcion': "publisher nulo → 'Unknown'", 'filas_antes': 16598, 'filas_despues': 16598, 'filas_afectadas': 58}, {'paso': '3.3b', 'descripcion': 'Eliminar filas sin year', 'filas_antes': 16598, 'filas_despues': 16327, 'filas_afectadas': 271}]


### 3.4 Quitar duplicados exactos (unicidad)

Si todas las columnas del dataset son iguales, la fila está repetida: se conserva la primera y se eliminan las demás. Comparamos solo `COLS_NEGOCIO`, porque los metadatos (`_row_hash`, `_batch_id`) siempre son distintos entre filas.

> Aunque el Paso 2 haya dado 0, la regla se deja en el pipeline: protege las siguientes ejecuciones, cuando lleguen datos nuevos.

In [18]:
# ── 3.4 Duplicados exactos ─────────────────────────────────────────────────

antes = len(df)
df = df.drop_duplicates(subset=COLS_NEGOCIO, keep='first')

registrar('3.4', 'Eliminar duplicados exactos', antes, len(df), antes - len(df))

[3.4] Eliminar duplicados exactos
       filas: 16,327 → 16,327   (afectadas: 0)


### 3.5 Rangos y consistencia (validez y consistencia)

| Regla | Acción si no se cumple |
|---|---|
| `year` entre `ANIO_MIN` y `ANIO_MAX` | **Eliminar**: es un error de captura |
| Ventas ≥ 0 | **Eliminar**: no existen ventas negativas |
| `global_sales` ≈ suma de las 4 regiones (± `TOLERANCIA`) | **Marcar** con `ventas_consistentes = False`, sin eliminar |

¿Por qué marcar y no eliminar la última? La fila puede ser real y el error estar solo en una columna. Eliminarla perdería información; marcarla deja que quien analice decida.

In [19]:
# ── 3.5 Rangos y consistencia ──────────────────────────────────────────────

# Años fuera de rango → eliminar
print()
antes = len(df)
df = df[df['year'].between(ANIO_MIN, ANIO_MAX)]
registrar('3.5a', f'Eliminar year fuera de [{ANIO_MIN}, {ANIO_MAX}]', antes, len(df), antes - len(df))




# Ventas negativas → eliminar (se conservan las filas con TODAS las ventas >= 0)
print()
antes = len(df)
df = df[(df[COLS_VENTAS] >= 0).all(axis=1)]
registrar('3.5b', 'Eliminar filas con ventas negativas', antes, len(df), antes - len(df))



# Consistencia → marcar, no eliminar
print()
diferencia = (df[COL_GLOBAL] - df[COLS_REGIONES].sum(axis=1)).abs()
df['ventas_consistentes'] = diferencia <= TOLERANCIA
marcadas = (~df['ventas_consistentes']).sum()
registrar('3.5c', f'Marcar ventas_consistentes (tolerancia ±{TOLERANCIA})', len(df), len(df), marcadas)
print()


[3.5a] Eliminar year fuera de [1980, 2016]
       filas: 16,327 → 16,323   (afectadas: 4)

[3.5b] Eliminar filas con ventas negativas
       filas: 16,323 → 16,323   (afectadas: 0)

[3.5c] Marcar ventas_consistentes (tolerancia ±0.05)
       filas: 16,323 → 16,323   (afectadas: 0)



In [20]:
df.head(5)

,rank,name,platform,year,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales,_batch_id,_extracted_at,_source_table,_row_hash,ventas_consistentes
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,1,2026-10-28T04:26:45,raw_df1,4afd4e8c1f96d38e,True
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,1,2026-10-28T04:26:45,raw_df1,3ee71d62807f116a,True
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,1,2026-10-28T04:26:45,raw_df1,688fb7c1f6ec1137,True
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,1,2026-10-28T04:26:45,raw_df1,a60698219af7ed96,True
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,1,2026-10-28T04:26:45,raw_df1,2644afa96579b93b,True


### 3.6 Convertir tipos de dato

SQLite guardó `year` como decimal (`2006.0`) porque tenía nulos, y la fecha de extracción como texto. Ahora que no hay nulos, cada columna toma su **tipo correcto**. Parquet guardará estos tipos dentro del archivo.

| Columna | Antes | Después |
|---|---|---|
| `year` | decimal (`float64`) | **entero** (`int64`) |
| ventas | decimal | decimal redondeado a 2 |
| `_extracted_at` | texto | **fecha y hora** |

https://pandas.pydata.org/docs/reference/arrays.html$0

In [21]:
# ── 3.6 Tipos de dato ──────────────────────────────────────────────────────

tipos_antes = df.dtypes

df['year']          = df['year'].astype('int64')
df[COLS_VENTAS]     = df[COLS_VENTAS].round(2)
df['_extracted_at'] = pd.to_datetime(df['_extracted_at']).astype('datetime64[us]')

registrar('3.6', 'Convertir tipos (year → entero; _extracted_at → fecha)', len(df), len(df), 0)

pd.DataFrame({'antes': tipos_antes, 'después': df.dtypes})

[3.6] Convertir tipos (year → entero; _extracted_at → fecha)
       filas: 16,323 → 16,323   (afectadas: 0)


,antes,después
rank,int64,int64
name,object,object
platform,object,object
year,float64,int64
genre,object,object
publisher,object,object
na_sales,float64,float64
eu_sales,float64,float64
jp_sales,float64,float64
other_sales,float64,float64


### 3.7 Enriquecer con un catálogo (*join*)

**Enriquecer** es agregar información que el dataset no trae. Unimos cada plataforma con su **fabricante** usando un catálogo pequeño (una tabla de referencia).

Dos cuidados en todo *join*:

1. **Que no multiplique filas.** `validate='m:1'` hace que pandas marque error si el catálogo tiene una plataforma repetida.
2. **Registros sin pareja.** Con `indicator=True` vemos qué plataformas no están en el catálogo; se les asigna `'Otro'`.

In [22]:
# ── 3.7 Catálogo de plataformas → fabricante ───────────────────────────────

catalogo_plataformas = pd.DataFrame([
    ('Wii', 'Nintendo'), ('WiiU', 'Nintendo'), ('NES', 'Nintendo'), ('SNES', 'Nintendo'),
    ('N64', 'Nintendo'), ('GC', 'Nintendo'), ('GB', 'Nintendo'), ('GBA', 'Nintendo'),
    ('DS', 'Nintendo'), ('3DS', 'Nintendo'),

    ('PS', 'Sony'), ('PS2', 'Sony'), ('PS3', 'Sony'), ('PS4', 'Sony'), ('PSP', 'Sony'), ('PSV', 'Sony'),

    ('XB', 'Microsoft'), ('X360', 'Microsoft'), ('XOne', 'Microsoft'),

    ('GEN', 'Sega'), ('SAT', 'Sega'), ('DC', 'Sega'), ('SCD', 'Sega'), ('GG', 'Sega'),

    ('2600', 'Atari'), ('PC', 'PC'),

    ('WS', 'Bandai'), ('NG', 'SNK'), ('TG16', 'NEC'), ('PCFX', 'NEC'), ('3DO', 'Panasonic'),

], columns=['platform', 'fabricante'])



antes = len(df)

df = df.merge(catalogo_plataformas, on='platform', how='left', validate='m:1', indicator=True)


# ¿Qué plataformas no encontraron fabricante?
sin_pareja = df[df['_merge'] == 'left_only']
df['fabricante'] = df['fabricante'].fillna('Otro')
df = df.drop(columns='_merge')

registrar('3.7', "Join con catálogo de plataformas (sin pareja → 'Otro')", antes, len(df), len(sin_pareja))


print()
print(f'Filas sin pareja en el catálogo: {len(sin_pareja)}')
print()

sin_pareja['platform'].value_counts()

[3.7] Join con catálogo de plataformas (sin pareja → 'Otro')
       filas: 16,323 → 16,323   (afectadas: 0)

Filas sin pareja en el catálogo: 0



,count
platform,


In [23]:
df.head(5)

,rank,name,platform,year,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales,_batch_id,_extracted_at,_source_table,_row_hash,ventas_consistentes,fabricante
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,1,2026-10-28 04:26:45,raw_df1,4afd4e8c1f96d38e,True,Nintendo
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,1,2026-10-28 04:26:45,raw_df1,3ee71d62807f116a,True,Nintendo
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,1,2026-10-28 04:26:45,raw_df1,688fb7c1f6ec1137,True,Nintendo
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,1,2026-10-28 04:26:45,raw_df1,a60698219af7ed96,True,Nintendo
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,1,2026-10-28 04:26:45,raw_df1,2644afa96579b93b,True,Nintendo


### 3.8 Columnas derivadas

Columnas calculadas a partir de otras que facilitan el análisis. `decada` permitirá agrupar sin repetir el cálculo en cada consulta.

In [24]:
# ── 3.8 Columnas derivadas ─────────────────────────────────────────────────

df['decada'] = df['year'] // 10 * 10          # 2006 → 2000

registrar('3.8', 'Columna derivada decada', len(df), len(df), 0)

df[['name', 'year', 'decada']].head()

[3.8] Columna derivada decada
       filas: 16,323 → 16,323   (afectadas: 0)


,name,year,decada
0,Wii Sports,2006,2000
1,Super Mario Bros.,1985,1980
2,Mario Kart Wii,2008,2000
3,Wii Sports Resort,2009,2000
4,Pokemon Red/Pokemon Blue,1996,1990


### 3.9 Metadatos de transformación

Igual que en EXTRACTED, agregamos cuándo y con qué versión de reglas se transformó cada fila. Los metadatos de extracción **se conservan**: con `_row_hash` cualquier fila limpia se rastrea hasta `extracted_df1` y `raw_df1`.

In [25]:
# ── 3.9 Metadatos y orden final de columnas ───────────────────────────────

df['_transformed_at']    = pd.Timestamp(datetime.now().replace(microsecond=0))
df['_transformed_at']    = df['_transformed_at'].astype('datetime64[us]')
df['_transform_version'] = VERSION_TRANSFORM



COLUMNAS_FINALES = [

    'rank', 'name', 'platform', 'fabricante', 'year', 'decada', 'genre', 'publisher',
    'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'global_sales', 'ventas_consistentes',
    '_batch_id', '_extracted_at', '_source_table', '_row_hash', '_transformed_at', '_transform_version',
]


df_final = df[COLUMNAS_FINALES].sort_values('rank').reset_index(drop=True)

registrar('3.9', 'Metadatos de transformación y orden de columnas', len(df_final), len(df_final), 0)



[3.9] Metadatos de transformación y orden de columnas
       filas: 16,323 → 16,323   (afectadas: 0)


In [26]:
inspeccionar_dataset(df_final, nombre=f'{NOMBRE_DESTINO} (antes de cargar)')


  DATASET: transformed_df1 (antes de cargar)
  Filas    : 16,323
  Columnas : 20

  Tipos de datos y valores nulos:
    rank                         int64        nulos: 0 (0.0%)
    name                         object       nulos: 0 (0.0%)
    platform                     object       nulos: 0 (0.0%)
    fabricante                   object       nulos: 0 (0.0%)
    year                         int64        nulos: 0 (0.0%)
    decada                       int64        nulos: 0 (0.0%)
    genre                        object       nulos: 0 (0.0%)
    publisher                    object       nulos: 0 (0.0%)
    na_sales                     float64      nulos: 0 (0.0%)
    eu_sales                     float64      nulos: 0 (0.0%)
    jp_sales                     float64      nulos: 0 (0.0%)
    other_sales                  float64      nulos: 0 (0.0%)
    global_sales                 float64      nulos: 0 (0.0%)
    ventas_consistentes          bool         nulos: 0 (0.0%)
    _batch_id  

,rank,name,platform,fabricante,year,decada,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales,ventas_consistentes,_batch_id,_extracted_at,_source_table,_row_hash,_transformed_at,_transform_version
0,1,Wii Sports,Wii,Nintendo,2006,2000,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,True,1,2026-10-28 04:26:45,raw_df1,4afd4e8c1f96d38e,2026-10-28 14:48:35,v1
1,2,Super Mario Bros.,NES,Nintendo,1985,1980,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,True,1,2026-10-28 04:26:45,raw_df1,3ee71d62807f116a,2026-10-28 14:48:35,v1
2,3,Mario Kart Wii,Wii,Nintendo,2008,2000,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,True,1,2026-10-28 04:26:45,raw_df1,688fb7c1f6ec1137,2026-10-28 14:48:35,v1


In [27]:
# ── La bitácora completa ───────────────────────────────────────────────────

df_bitacora = pd.DataFrame(bitacora)
df_bitacora

,paso,descripcion,filas_antes,filas_despues,filas_afectadas
0,3.1,Nombres de columnas a snake_case,16598,16598,0
1,3.2,"Quitar espacios extra en ['name', 'platform', ...",16598,16598,5
2,3.3a,publisher nulo → 'Unknown',16598,16598,58
3,3.3b,Eliminar filas sin year,16598,16327,271
4,3.4,Eliminar duplicados exactos,16327,16327,0
5,3.5a,"Eliminar year fuera de [1980, 2016]",16327,16323,4
6,3.5b,Eliminar filas con ventas negativas,16323,16323,0
7,3.5c,Marcar ventas_consistentes (tolerancia ±0.05),16323,16323,0
8,3.6,Convertir tipos (year → entero; _extracted_at ...,16323,16323,0
9,3.7,Join con catálogo de plataformas (sin pareja →...,16323,16323,0


---
## PASO 4 — Diagnóstico de calidad (después)

Repetimos **las mismas mediciones** del Paso 2, ahora sobre `df_final`. Las celdas son iguales; solo cambian el DataFrame y los nombres de columnas (ya en minúsculas).

In [28]:
# ── Nulos ──────────────────────────────────────────────────────────────────

nulos_d = df_final[COLS_NEGOCIO].isna().sum()

print()
print(f'Total de celdas nulas: {nulos_d.sum():,}')
print()


Total de celdas nulas: 0



In [29]:
# ── Duplicados exactos ─────────────────────────────────────────────────────

dup_exactos_d = df_final.duplicated(subset=COLS_NEGOCIO).sum()

print()
print(f'Duplicados exactos: {dup_exactos_d}')
print()


Duplicados exactos: 0



In [30]:
# ── Validez ────────────────────────────────────────────────────────────────

anio_invalido_d  = ~df_final['year'].between(ANIO_MIN, ANIO_MAX)
venta_negativa_d = (df_final[COLS_VENTAS] < 0).any(axis=1)

print()
print(f'Años fuera de [{ANIO_MIN}, {ANIO_MAX}] : {anio_invalido_d.sum()}')
print(f'Filas con ventas negativas    : {venta_negativa_d.sum()}')
print()


Años fuera de [1980, 2016] : 0
Filas con ventas negativas    : 0



In [31]:
# ── Consistencia ───────────────────────────────────────────────────────────

diferencia_d    = (df_final[COL_GLOBAL] - df_final[COLS_REGIONES].sum(axis=1)).abs()
inconsistente_d = diferencia_d > TOLERANCIA

print()
print(f'Filas inconsistentes (diferencia > {TOLERANCIA}) : {inconsistente_d.sum()}')
print()


Filas inconsistentes (diferencia > 0.05) : 0



In [32]:
# ── Antes vs. después ──────────────────────────────────────────────────────

calidad_despues = pd.Series({
    'filas'                  : len(df_final),
    'celdas_nulas'           : nulos_d.sum(),
    'duplicados_exactos'     : dup_exactos_d,
    'anios_fuera_de_rango'   : anio_invalido_d.sum(),
    'ventas_negativas'       : venta_negativa_d.sum(),
    'ventas_inconsistentes'  : inconsistente_d.sum(),
}, name='después')

df_calidad = pd.DataFrame({'antes': calidad_antes, 'después': calidad_despues})
df_calidad['cambio'] = df_calidad['después'] - df_calidad['antes']
df_calidad

,antes,después,cambio
filas,16598,16323,-275
celdas_nulas,329,0,-329
duplicados_exactos,0,0,0
anios_fuera_de_rango,4,0,-4
ventas_negativas,0,0,0
ventas_inconsistentes,0,0,0


**Cómo leerla:**
- Nulos, duplicados, años fuera de rango y ventas negativas deben quedar en **0**.
- `ventas_inconsistentes` puede seguir arriba de 0: esas filas se **marcaron** (`ventas_consistentes = False`), no se eliminaron.
- La baja de `filas` debe coincidir con la suma de filas eliminadas en la bitácora.

In [33]:
# ── ¿Cuadra la bitácora con el conteo de filas? ────────────────────────────

eliminadas_bitacora = (df_bitacora['filas_antes'] - df_bitacora['filas_despues']).sum()
eliminadas_real     = len(df_ext) - len(df_final)

print()
print(f'Filas eliminadas según la bitácora : {eliminadas_bitacora:,}')
print(f'Filas eliminadas en realidad       : {eliminadas_real:,}')

if eliminadas_bitacora == eliminadas_real:
    print('Cuadra')
else:
    print('NO CUADRA: revisa la bitácora')
print()


Filas eliminadas según la bitácora : 275
Filas eliminadas en realidad       : 275
Cuadra



---
## PASO 5 — Carga: escribir la capa TRANSFORMED en Parquet

Hasta aquí, `df_final` **solo vive en memoria**: si cierras Colab, se pierde. La **carga** (la "L" de ETL) lo guarda en un destino persistente para que otros, y la Unidad 3, lo consuman.

```
extracted_df1 (SQLite)  →  transformaciones  →  transformed_df1.parquet  (Drive)
```

### ¿Por qué Parquet y no el `.db`?

- En la industria, la capa limpia (*silver*) vive normalmente en archivos **Parquet** dentro de un *data lake*. La base de datos para análisis llega después, al modelar (Unidad 3).
- Parquet **guarda el tipo de cada columna** dentro del archivo: lo que hicimos en 3.6 no se pierde. SQLite y CSV no lo conservan.



### ¿Append o sobrescribir?

| Estrategia | Qué hace | ¿Dónde la usamos? |
|---|---|---|
| **Append** | Agrega filas nuevas al final | EXTRACTED: cada lote es historia que no se puede perder |
| **Sobrescribir** (*full refresh*) | Reemplaza el archivo completo | **TRANSFORMED**: se puede recalcular entero desde EXTRACTED |

Aquí **sobrescribimos**. Si mañana cambia una regla de limpieza, basta con volver a ejecutar el notebook. A esto se le llama **idempotencia**: ejecutar el pipeline una o diez veces deja el mismo resultado.

In [34]:
# ── Librerías y rutas de la carga ──────────────────────────────────────────

import os
import json

TRANSFORMACION_PATH = f'{PROYECTO_BASE}/Transformacion_de_Datos'
PROCESSED_PATH      = f'{TRANSFORMACION_PATH}/Processed'     # aquí se guardan los Parquet
LOGS_PATH           = f'{TRANSFORMACION_PATH}/Logs'          # log y lineage
REPORTES_PATH       = f'{TRANSFORMACION_PATH}/Reportes'      # evidencias para el reporte


os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(LOGS_PATH, exist_ok=True)
os.makedirs(REPORTES_PATH, exist_ok=True)


PARQUET_DESTINO = f'{PROCESSED_PATH}/{NOMBRE_DESTINO}.parquet'
LOG_PATH        = f'{LOGS_PATH}/transformacion_log.jsonl'
LINEAGE_PATH    = f'{LOGS_PATH}/lineage_{NOMBRE_DESTINO}.json'

print()
print(f'Destino : {PARQUET_DESTINO}')
print()


Destino : /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Processed/transformed_df1.parquet



In [35]:
# ── Log de operaciones (igual que en EXTRACCIÓN: una línea JSON por operación) ──

def escribir_log(etapa, filas_entrada, filas_salida, estado='exitoso', detalle=None):

    '''Agrega UNA línea al log de transformación.'''


    evento = {

        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'etapa'        : etapa,
        'origen'       : TABLA_EXTRACTED,
        'destino'      : PARQUET_DESTINO,
        'filas_entrada': filas_entrada,
        'filas_salida' : filas_salida,
        'version'      : VERSION_TRANSFORM,
        'estado'       : estado,
        'detalle'      : detalle,
    }

    with open(LOG_PATH, 'a', encoding='utf-8') as f:      # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')


print()
print('escribir_log() lista')
print()


escribir_log() lista



In [36]:
# ── Cargar df_final a Parquet ──────────────────────────────────────────────

df_final.to_parquet(PARQUET_DESTINO, index=False)        # si el archivo existe, se sobrescribe

tamanio_mb = os.path.getsize(PARQUET_DESTINO) / 1_048_576

escribir_log('carga_parquet', len(df_ext), len(df_final), detalle=f'{tamanio_mb:.2f} MB')


print()
print(f'{TABLA_EXTRACTED} → {NOMBRE_DESTINO}.parquet')
print(f'  Filas de entrada : {len(df_ext):,}')
print(f'  Filas cargadas   : {len(df_final):,}')
print(f'  Tamaño           : {tamanio_mb:.2f} MB')
print()


extracted_df1 → transformed_df1.parquet
  Filas de entrada : 16,598
  Filas cargadas   : 16,323
  Tamaño           : 0.75 MB



### ¿Se conservaron los tipos?

Leemos el archivo de vuelta y comparamos sus tipos con los de `df_final`. Deben ser **iguales**: esa es la ventaja de Parquet.

In [37]:
# ── Leer el Parquet de vuelta ──────────────────────────────────────────────

df_leido = pd.read_parquet(PARQUET_DESTINO)

print()
print(f'Filas leídas: {len(df_leido):,}')
print()

pd.DataFrame({'en memoria': df_final.dtypes, 'desde Parquet': df_leido.dtypes})


Filas leídas: 16,323



,en memoria,desde Parquet
rank,int64,int64
name,object,object
platform,object,object
fabricante,object,object
year,int64,int64
decada,int64,int64
genre,object,object
publisher,object,object
na_sales,float64,float64
eu_sales,float64,float64


---
## PASO 6 — Lineage y evidencias

El **lineage** documenta el recorrido completo del dato: **origen → transformaciones → destino**. Lo guardamos como JSON junto al log. Cada fila, además, conserva su `_row_hash`, que la vincula con `extracted_df1` y `raw_df1`.

También guardamos en `Reportes/` las tablas que van en el **reporte de validación**: la bitácora y la comparación antes vs. después.

In [38]:
# ── Guardar el lineage ─────────────────────────────────────────────────────

lineage = {

    'recorrido'        : ['Kaggle (CSV)', 'raw_df1 (SQLite)', f'{TABLA_EXTRACTED} (SQLite)', f'{NOMBRE_DESTINO} (Parquet)'],
    'origen'           : TABLA_EXTRACTED,
    'base_de_datos'    : DB_PATH,
    'filas_origen'     : len(df_ext),
    'destino'          : PARQUET_DESTINO,
    'filas_destino'    : len(df_final),
    'estrategia_carga' : 'sobrescribir (full refresh)',
    'version_reglas'   : VERSION_TRANSFORM,
    'generado_en'      : datetime.now().isoformat(timespec='seconds'),
    'trazabilidad'     : '_row_hash vincula cada fila con extracted_df1 y raw_df1',
    'transformaciones' : bitacora,                                    # la bitácora del Paso 3
    'columnas_destino' : df_final.dtypes.astype(str).to_dict(),       # {columna: tipo}
}

with open(LINEAGE_PATH, 'w', encoding='utf-8') as f:
    json.dump(lineage, f, indent=2, ensure_ascii=False, default=int)

print()
print(f'Lineage guardado en: {LINEAGE_PATH}')
print()
print(' → '.join(lineage['recorrido']))
print()


Lineage guardado en: /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Logs/lineage_transformed_df1.json

Kaggle (CSV) → raw_df1 (SQLite) → extracted_df1 (SQLite) → transformed_df1 (Parquet)



> `default=int` convierte a entero los números de pandas/NumPy (por ejemplo, un `int64`), que el módulo `json` no sabe escribir por sí solo.

In [39]:
# ── Guardar las evidencias para el reporte ─────────────────────────────────

ruta_bitacora = f'{REPORTES_PATH}/bitacora_{NOMBRE_DESTINO}.csv'
ruta_calidad  = f'{REPORTES_PATH}/calidad_{NOMBRE_DESTINO}.csv'


df_bitacora.to_csv(ruta_bitacora, index=False, encoding='utf-8-sig')       # utf-8-sig: abre bien en Excel
df_calidad.to_csv(ruta_calidad, index_label='metrica', encoding='utf-8-sig')


print()
print('Evidencias guardadas:')
print(f'  {ruta_bitacora}')
print(f'  {ruta_calidad}')
print()




Evidencias guardadas:
  /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Reportes/bitacora_transformed_df1.csv
  /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Reportes/calidad_transformed_df1.csv



---
## PASO 7 — Validación con DuckDB (sobre el Parquet ya cargado)

Una carga no está terminada hasta que **se valida lo que quedó en el destino**, no lo que había en memoria. Con **DuckDB** consultamos el archivo Parquet directamente con SQL, sin cargarlo a ninguna base.



In [40]:
# ── DuckDB ─────────────────────────────────────────────────────────────────
# En Colab suele venir instalado; si no, quita el # de la siguiente línea:
# !pip install duckdb

import duckdb

con = duckdb.connect()        # base en memoria: no crea ningún archivo

P = f"read_parquet('{PARQUET_DESTINO}')"         # así se lee un Parquet dentro de SQL

print()
print(f'DuckDB {duckdb.__version__} listo')
print()


DuckDB 1.3.2 listo



In [41]:
# ── Primer vistazo: SQL directo sobre el archivo ───────────────────────────

con.sql(f'''
    SELECT name, platform, fabricante, year, global_sales
    FROM {P}
    ORDER BY global_sales DESC
    LIMIT 5
''').df()

,name,platform,fabricante,year,global_sales
0,Wii Sports,Wii,Nintendo,2006,82.74
1,Super Mario Bros.,NES,Nintendo,1985,40.24
2,Mario Kart Wii,Wii,Nintendo,2008,35.82
3,Wii Sports Resort,Wii,Nintendo,2009,33.00
4,Pokemon Red/Pokemon Blue,GB,Nintendo,1996,31.37


In [42]:
# ── Esquema: columnas y tipos guardados en el archivo ──────────────────────

con.sql(f'DESCRIBE SELECT * FROM {P}').df()

,column_name,column_type,null,key,default,extra
0,rank,BIGINT,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,platform,VARCHAR,YES,None,None,None
3,fabricante,VARCHAR,YES,None,None,None
4,year,BIGINT,YES,None,None,None
5,decada,BIGINT,YES,None,None,None
6,genre,VARCHAR,YES,None,None,None
7,publisher,VARCHAR,YES,None,None,None
8,na_sales,DOUBLE,YES,None,None,None
9,eu_sales,DOUBLE,YES,None,None,None


In [43]:
# ── Conteo y nulos en el archivo cargado ───────────────────────────────────

con.sql(f'''
    SELECT COUNT(*)                                           AS filas,
           SUM(CASE WHEN year      IS NULL THEN 1 ELSE 0 END) AS nulos_year,
           SUM(CASE WHEN publisher IS NULL THEN 1 ELSE 0 END) AS nulos_publisher
    FROM {P}
''').df()

,filas,nulos_year,nulos_publisher
0,16323,0.0,0.0


In [44]:
# ── Cerrar la conexión de DuckDB ───────────────────────────────────────────

con.close()

print()
print('Conexión cerrada')
print()


Conexión cerrada

